# 06. 클래스(OOP) 기초
**목표**: 분석 코드를 `DataAnalyzer` 클래스로 묶는 감각을 익힙니다. (예상 3~4시간)

## 왜 클래스로 만들까?
노트북에서 셀을 이어 붙인 분석은 **재사용하기 어렵습니다.** 클래스로 만들면
- 데이터를 `self.df` 에 한 번 담아 두고 여러 메서드가 이어서 사용
- 다른 데이터에도 같은 분석을 즉시 적용
- 파라미터(임계값, 전략)를 인자로 바꿔 가며 실험

## 1. 문법 최소 정리
- `class 이름:` 으로 정의
- `__init__(self, ...)`: 객체를 만들 때 한 번 실행되는 초기화 메서드
- `self`: "이 객체 자신". 객체 안의 데이터는 `self.xxx` 에 저장
- 메서드: 클래스 안에 정의된 함수. 첫 인자는 항상 `self`

In [ ]:
class Counter:
    def __init__(self, start=0):
        self.value = start          # 객체의 상태

    def add(self, n=1):
        self.value += n
        return self                 # 자기 자신을 돌려주면 메서드를 이어 쓸 수 있음

c = Counter()
c.add().add(5)
print(c.value)   # 6

## 2. 미니 DataAnalyzer 만들어 보기
미션 8장의 뼈대를 실제로 동작하는 형태로 채워 봅니다.
데이터는 CSV 대신 DataFrame을 바로 받을 수 있게 해서 노트북에서 시험하기 쉽게 했습니다.

In [ ]:
import numpy as np
import pandas as pd


class MiniAnalyzer:
    def __init__(self, df: pd.DataFrame | None = None, data_path: str | None = None):
        self.data_path = data_path
        self.df = df

    # ---- 1) 로드
    def load_data(self) -> pd.DataFrame:
        if self.df is None:
            self.df = pd.read_csv(self.data_path)
        return self.df

    # ---- 2) 결측치 (전략을 인자로)
    def handle_missing_values(self, column: str, strategy: str = "group_median",
                              group_col: str | None = None) -> pd.DataFrame:
        s = self.df[column]
        if strategy == "mean":
            self.df[column] = s.fillna(s.mean())
        elif strategy == "median":
            self.df[column] = s.fillna(s.median())
        elif strategy == "group_median":
            if group_col is None:
                raise ValueError("group_median 전략은 group_col이 필요합니다")
            self.df[column] = s.fillna(self.df.groupby(group_col)[column].transform("median"))
        else:
            raise ValueError(f"알 수 없는 전략: {strategy}")
        return self.df

    # ---- 3) 이상치 (임계값을 인자로)
    def detect_outliers(self, column: str, threshold: float = 1.5) -> pd.DataFrame:
        s = self.df[column]
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - threshold * iqr, q3 + threshold * iqr
        return self.df[(s < lower) | (s > upper)]

    # ---- 4) RFM
    def calculate_rfm(self, customer_col: str, date_col: str, amount_col: str) -> pd.DataFrame:
        snapshot = self.df[date_col].max() + pd.Timedelta(days=1)
        return self.df.groupby(customer_col).agg(
            recency=(date_col, lambda s: (snapshot - s.max()).days),
            frequency=(date_col, "count"),
            monetary=(amount_col, "sum"),
        )

## 3. 사용해 보기

In [ ]:
rng = np.random.default_rng(5)
n = 120
raw = pd.DataFrame({
    "customer_id": rng.integers(1, 21, n),
    "category": rng.choice(["상의", "하의", "신발"], n),
    "order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "amount": rng.normal(60000, 15000, n).round(-2),
})
raw.loc[[2, 9, 30], "amount"] = np.nan
raw.loc[5, "amount"] = 2_000_000    # 이상치

an = MiniAnalyzer(df=raw)
an.load_data()
print("결측 전:", an.df["amount"].isna().sum())

an.handle_missing_values("amount", strategy="group_median", group_col="category")
print("결측 후:", an.df["amount"].isna().sum())

print("이상치:")
display(an.detect_outliers("amount", threshold=1.5))

In [ ]:
an.calculate_rfm("customer_id", "order_date", "amount").head()

## 4. 설계 팁
- **하나의 메서드는 하나의 일**: 결측 처리와 이상치 탐지를 한 메서드에 섞지 않습니다.
- **파라미터는 인자로**: 임계값(`threshold`), 전략(`strategy`)을 코드 안에 고정하지 않습니다. (미션 4.2)
- **원본 보존이 필요하면** `self.raw_df = df.copy()` 를 따로 둡니다.
- **타입 힌트와 docstring** 으로 인자 의미를 적어 두면 나중의 나에게도 도움이 됩니다.

## 5. 실제 미션으로 옮길 때의 흐름
1. 노트북에서 실험 (`02`~`05` 처럼)
2. 잘 되는 코드를 메서드로 이동 → `src/pipeline.py`
3. 노트북에서는 `from src.pipeline import DataAnalyzer` 로 불러서 호출만 하기 (경로 문제가 생기면 `sys.path` 에 프로젝트 루트를 추가)
4. 미션 필수 메서드 4개 + 이미지/텍스트 피처 메서드 추가

### ✏️ 직접 해보기
`MiniAnalyzer` 에 텍스트 피처 메서드 `add_text_features(self, column)` 를 만들어 단어 수와 문자열 길이 컬럼을 추가하세요. (`02_pandas_basics` 6번 참고)

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [ ]:
def add_text_features(self, column):
    self.df[f"{column}_word_count"] = self.df[column].str.split().str.len()
    self.df[f"{column}_len"] = self.df[column].str.len()
    return self.df

MiniAnalyzer.add_text_features = add_text_features   # 노트북에서 빠르게 확인하기 위해 클래스에 붙임

t = MiniAnalyzer(df=pd.DataFrame({"name": ["Red T-Shirt", "Hat"]}))
t.add_text_features("name")

### ✏️ 직접 해보기
`MiniAnalyzer` 에 이미지 피처 메서드 `add_image_features(self, images)` 를 만들어 보세요. `images` 는 `(N, H, W)` 배열이며 행 수 N과 `self.df` 가 같다고 가정합니다. for문 금지.

> 힌트: `01_numpy_basics` 의 `axis=(1, 2)`

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [ ]:
def add_image_features(self, images):
    self.df["img_mean"] = images.mean(axis=(1, 2))
    self.df["img_std"] = images.std(axis=(1, 2))
    return self.df

MiniAnalyzer.add_image_features = add_image_features

imgs = np.random.default_rng(0).integers(0, 256, size=(2, 8, 8))
t.add_image_features(imgs)

---
## 🎉 학습 단계 완료
여기까지 마쳤다면 본 미션을 시작할 준비가 된 것입니다. 추천 순서는 다음과 같습니다.

1. 데이터셋 선정 (1,000건 이상, 8컬럼 이상, 거래 정보 + 이미지)
2. `analysis_report.ipynb` 에서 먼저 실험
3. 코드를 `src/pipeline.py` 의 `DataAnalyzer` 로 이동
4. 해석 마크다운과 README 인사이트 작성